# SignSight — YOLO trên Colab / Kaggle
Chọn **Runtime → Change runtime type → GPU (T4)** trên Colab, hoặc bật GPU + Internet trên Kaggle.
Notebook chứa code, taxonomy và các CSV train/val/test hiện tại. Không cần clone repo hoặc upload ZIP.
Dùng cùng nguồn GTSRB như notebook CNN: ưu tiên dataset đã gắn ở `/kaggle/input`, nếu thiếu thì tải bằng KaggleHub.
Bật `USE_DRIVE = True` trên Colab để lưu dataset YOLO đã chuẩn bị và checkpoint qua các phiên runtime.
GTSRB ROI là baseline kỹ thuật; cần dữ liệu cảnh đường phố Việt Nam để đánh giá triển khai.
Checkpoint và metric detection độc lập với CNN best_model.pth / accuracy 99,47%.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ultralytics==8.4.173', 'PyYAML>=6,<7', 'kagglehub'])
import torch
assert torch.cuda.is_available(), 'Chọn GPU runtime trước khi train.'
print(torch.__version__, torch.cuda.get_device_name(0))

## Code và splits có sẵn
Giữ nguyên split theo track và taxonomy 10 lớp của repo, không chia lại dữ liệu khi mở notebook.
Cache được định danh theo nguồn, CSV, code converter/validator và taxonomy; thay đổi đầu vào sẽ tạo cache mới.

In [ ]:
#@title Khôi phục pipeline và splits có sẵn (không cần sửa dữ liệu nhúng)
from pathlib import Path
import json, os, shutil
IS_KAGGLE = Path('/kaggle/input').is_dir()
WORK = Path('/kaggle/working/signsight_yolo' if IS_KAGGLE else '/content/signsight_yolo')
WORK.mkdir(parents=True, exist_ok=True)
os.environ['YOLO_CONFIG_DIR'] = str(WORK / '.cache/ultralytics')
os.environ['YOLO_AUTOINSTALL'] = 'false'
os.environ['KAGGLEHUB_CACHE'] = str(WORK / '.cache/kagglehub')
TRAINING_FILES = {'ai_engine/config.py': 'import os\nimport json\n\nPROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\nDATA_DIR = os.path.join(PROJECT_ROOT, "data")\nTAXONOMY_PATH = os.path.join(DATA_DIR, "taxonomy.json")\n\n# Model & Training parameters\nNUM_CLASSES = 10\nIMAGE_SIZE = (64, 64)  # High resolution enough for sign symbols, lightweight for edge/web\nBATCH_SIZE = 64\nLEARNING_RATE = 1e-3\nNUM_EPOCHS = 15\nCONFIDENCE_THRESHOLD = 0.70  # Below this, sign is flagged as \'is_uncertain\'\n\n# Expiration policies per sign type (in days)\nEXPIRATION_POLICIES = {\n    "W.227": 14,      # Temporary road work (short lifecycle)\n    "W.245a": 60,     # General caution / road hazard\n    "P.125": 180,     # No overtaking (semi-permanent)\n    "P.127 (50)": 365,# Speed limit 50 km/h (permanent)\n    "P.127 (60)": 365,# Speed limit 60 km/h (permanent)\n    "P.127 (80)": 365,# Speed limit 80 km/h (permanent)\n    "P.122": 365,     # STOP sign (permanent)\n    "P.102": 365,     # No entry (permanent)\n    "R.301a": 365,    # Ahead only (permanent)\n    "R.301e": 365     # Keep right (permanent)\n}\n\n# Duplicate detection threshold\nDUPLICATE_DISTANCE_THRESHOLD_METERS = 30.0  # meters\n\ndef load_taxonomy():\n    if os.path.exists(TAXONOMY_PATH):\n        with open(TAXONOMY_PATH, "r", encoding="utf-8") as f:\n            return json.load(f)\n    return []\n\nTAXONOMY = load_taxonomy()\nCLASS_MAPPING = {item["new_id"]: item for item in TAXONOMY}\n', 'ai_engine/catalog.py': '"""Stable application catalog IDs, separate from checkpoint-specific class indices."""\nfrom copy import deepcopy\nimport hashlib\nimport json\nfrom pathlib import Path\nfrom types import MappingProxyType\n\nfrom ai_engine.config import TAXONOMY\n\n\ndef json_hash(value):\n    return hashlib.sha256(json.dumps(value,ensure_ascii=False,sort_keys=True,\n                                    separators=(\',\',\':\'),allow_nan=False).encode()).hexdigest()\n\n\nclass SignCatalog:\n    def __init__(self,items=None):\n        items=deepcopy(TAXONOMY if items is None else items)\n        if not isinstance(items,list) or not 1 <= len(items) <= 1000:\n            raise ValueError(\'Catalog requires 1..1000 entries\')\n        by_id={};codes=set()\n        for item in items:\n            if (not isinstance(item,dict) or type(item.get(\'new_id\')) is not int or\n                not 0 <= item[\'new_id\'] <= 1000000 or\n                any(not isinstance(item.get(k),str) or not item[k].strip() or len(item[k])>limit\n                    for k,limit in ((\'qcvn_code\',20),(\'name_vi\',255)))):\n                raise ValueError(\'Invalid catalog entry\')\n            if item[\'new_id\'] in by_id or item[\'qcvn_code\'] in codes:\n                raise ValueError(\'Duplicate catalog ID or code\')\n            by_id[item[\'new_id\']]=item;codes.add(item[\'qcvn_code\'])\n        for legacy in TAXONOMY:\n            existing=by_id.get(legacy[\'new_id\'])\n            if existing is None or any(existing[k]!=legacy[k] for k in (\'qcvn_code\',\'name_vi\')):\n                raise ValueError(\'Catalog must preserve legacy IDs, codes and names; use an explicit data migration to revise them\')\n        self.items=tuple(MappingProxyType(by_id[key]) for key in sorted(by_id))\n        self.by_id=MappingProxyType({item[\'new_id\']:item for item in self.items})\n        self.sha256=json_hash([dict(item) for item in self.items])\n\n    @classmethod\n    def from_file(cls,path=None):\n        return cls() if path is None else cls(json.loads(Path(path).read_text(encoding=\'utf-8-sig\')))\n\n    def public_items(self):\n        return [{key:item[key] for key in (\'new_id\',\'qcvn_code\',\'name_vi\')} for item in self.items]\n\n\nDEFAULT_CATALOG=SignCatalog()\n\n\nclass ModelClassMapping:\n    def __init__(self,document,catalog,checkpoint):\n        if not isinstance(document,dict) or document.get(\'schema_version\')!=1:\n            raise ValueError(\'Unsupported model mapping schema\')\n        if document.get(\'catalog_sha256\')!=catalog.sha256:\n            raise ValueError(\'Mapping catalog hash mismatch\')\n        with Path(checkpoint).open(\'rb\') as stream:\n            checksum=hashlib.file_digest(stream,\'sha256\').hexdigest()\n        if document.get(\'checkpoint_sha256\')!=checksum:\n            raise ValueError(\'Mapping checkpoint hash mismatch\')\n        entries=document.get(\'classes\')\n        if not isinstance(entries,list) or not 1 <= len(entries) <= 1000:\n            raise ValueError(\'Mapping requires an entry for every model class\')\n        names=[];targets=[]\n        for index,item in enumerate(entries):\n            if (not isinstance(item,dict) or type(item.get(\'model_class_id\')) is not int or\n                item[\'model_class_id\']!=index or item.get(\'reviewed\') is not True or\n                not isinstance(item.get(\'model_name\'),str) or not item[\'model_name\'].strip() or\n                type(item.get(\'catalog_id\')) is not int or item[\'catalog_id\'] not in catalog.by_id):\n                raise ValueError(\'Mapping entries must be ordered, reviewed and reference known catalog IDs\')\n            names.append(item[\'model_name\']);targets.append(item[\'catalog_id\'])\n        if len(set(names))!=len(names) or len(set(targets))!=len(targets):\n            raise ValueError(\'Duplicate model name or catalog target in mapping\')\n        self.names=names;self.targets=tuple(targets);self.sha256=json_hash(document)\n\n    def catalog_id(self,model_class_id):\n        if type(model_class_id) is not int or not 0 <= model_class_id < len(self.targets):\n            raise ValueError(\'Unsupported model class index\')\n        return self.targets[model_class_id]\n', 'ai_engine/detection.py': '"""Optional YOLO detector. Missing/incompatible weights never fall back to CNN."""\n\nimport hashlib\nimport math\nimport os\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom threading import Lock\n\nfrom ai_engine.config import CLASS_MAPPING, PROJECT_ROOT\nfrom ai_engine.catalog import DEFAULT_CATALOG, ModelClassMapping\n\n\nclass DetectorUnavailable(RuntimeError):\n    pass\n\n\n@dataclass\nclass Prediction:\n    predicted_class: int\n    confidence: float\n    bbox: tuple[float, float, float, float] | None\n    model_class_id: int | None = None\n\n\n@dataclass\nclass DetectionResult:\n    model_version: str\n    predictions: list[Prediction]\n    mapping_sha256: str | None = None\n    catalog_sha256: str | None = None\n\n\ndef checkpoint_hash(path: Path) -> str:\n    path = Path(path)\n    checksum = hashlib.sha256()\n    with path.open("rb") as source:\n        for chunk in iter(lambda: source.read(1024 * 1024), b""):\n            checksum.update(chunk)\n    return "sha256:" + checksum.hexdigest()\n\n\ndef configure_ultralytics():\n    config_dir = os.environ.setdefault("YOLO_CONFIG_DIR", str(Path(PROJECT_ROOT) / ".cache" / "ultralytics"))\n    Path(config_dir).mkdir(parents=True, exist_ok=True)\n    os.environ.setdefault("YOLO_AUTOINSTALL", "false")\n\n\nclass YoloDetector:\n    def __init__(self, weights: Path, device: str | None = None, confidence: float = 0.25,\n                 *, catalog=None, mapping_path=None):\n        if not math.isfinite(confidence) or not 0 < confidence <= 1:\n            raise ValueError("YOLO confidence must be finite and greater than 0, up to 1")\n        self.weights = Path(weights)\n        self.device = device\n        self.confidence = confidence\n        self.catalog = catalog or DEFAULT_CATALOG\n        self.mapping_path = Path(mapping_path) if mapping_path is not None else None\n        self._mapping = None\n        self._lock = Lock()\n        self._model = None\n        self._signature = None\n        self._version = None\n        self._reason = "weights_missing"\n\n    def _load(self):\n        if not self.weights.is_file():\n            self._model = None\n            self._reason = "weights_missing"\n            raise DetectorUnavailable("YOLO checkpoint is not configured")\n        mapping_bytes = None\n        if self.mapping_path is not None:\n            try:\n                if self.mapping_path.stat().st_size > 1024*1024: raise ValueError(\'Mapping too large\')\n                mapping_bytes = self.mapping_path.read_bytes()\n            except (OSError,ValueError):\n                self._model = None; self._reason = \'mapping_invalid\'\n                raise DetectorUnavailable(\'YOLO model mapping is missing or invalid\') from None\n        signature = (self.weights.stat().st_mtime_ns, self.weights.stat().st_size,\n                     hashlib.sha256(mapping_bytes).hexdigest() if mapping_bytes is not None else None)\n        if self._model is not None and signature == self._signature:\n            return\n        configure_ultralytics()\n        try:\n            from ultralytics import YOLO\n        except ImportError:\n            self._reason = "dependencies_missing"\n            raise DetectorUnavailable("Install ai_engine/requirements-yolo.txt to use YOLO") from None\n        try:\n            mapping = None\n            if mapping_bytes is not None:\n                try: mapping = ModelClassMapping(json.loads(mapping_bytes),self.catalog,self.weights)\n                except (ValueError,TypeError,KeyError):\n                    self._reason = \'mapping_invalid\'\n                    raise DetectorUnavailable(\'YOLO mapping must be reviewed and match this checkpoint/catalog\') from None\n            model = YOLO(str(self.weights), task="detect")\n            names = model.names\n            names = [names[i] for i in range(len(names))] if isinstance(names, dict) else list(names)\n            expected = mapping.names if mapping else [CLASS_MAPPING[i]["qcvn_code"] for i in range(10)]\n            if model.task != "detect" or names != expected:\n                self._reason = "taxonomy_mismatch"\n                raise DetectorUnavailable("YOLO checkpoint taxonomy must match the configured reviewed mapping or legacy taxonomy")\n            version = checkpoint_hash(self.weights)\n        except DetectorUnavailable:\n            self._model = None\n            raise\n        except Exception:\n            self._model = None\n            self._reason = "invalid_checkpoint"\n            raise DetectorUnavailable("YOLO checkpoint could not be loaded") from None\n        self._model, self._version, self._signature = model, version, signature\n        self._mapping = mapping\n        self._reason = None\n\n    def status(self):\n        with self._lock:\n            try:\n                self._load()\n            except DetectorUnavailable:\n                return {"ready": False, "reason": self._reason, "model_version": None,\n                        "confidence_threshold": self.confidence}\n            return {"ready": True, "reason": None, "model_version": self._version,\n                    "confidence_threshold": self.confidence,\n                    \'mapping_sha256\': self._mapping.sha256 if self._mapping else None,\n                    \'catalog_sha256\': self.catalog.sha256}\n\n    def predict(self, image) -> DetectionResult:\n        with self._lock:\n            self._load()\n            try:\n                results = self._model.predict(source=image, conf=self.confidence, iou=0.45, max_det=25,\n                                              device=self.device, save=False, verbose=False)\n            except Exception:\n                raise DetectorUnavailable("YOLO inference failed; check the configured device and checkpoint") from None\n            predictions = []\n            for result in results:\n                if result.boxes is None:\n                    continue\n                for coords, score, class_id in zip(result.boxes.xyxy.cpu().tolist(),\n                                                   result.boxes.conf.cpu().tolist(), result.boxes.cls.cpu().tolist()):\n                    maximum = len(self._mapping.targets) if self._mapping else len(CLASS_MAPPING)\n                    if not math.isfinite(class_id) or class_id != int(class_id) or not 0 <= int(class_id) < maximum:\n                        raise DetectorUnavailable("Detector returned an unsupported class")\n                    if not math.isfinite(score) or not 0 <= score <= 1 or not all(math.isfinite(value) for value in coords):\n                        raise DetectorUnavailable("Detector returned invalid coordinates or confidence")\n                    x1, y1, x2, y2 = coords\n                    box = (max(0.0, min(x1, image.width)), max(0.0, min(y1, image.height)),\n                           max(0.0, min(x2, image.width)), max(0.0, min(y2, image.height)))\n                    if box[2] <= box[0] or box[3] <= box[1]:\n                        raise DetectorUnavailable("Detector returned an empty bounding box")\n                    raw_id = int(class_id)\n                    catalog_id = self._mapping.catalog_id(raw_id) if self._mapping else raw_id\n                    predictions.append(Prediction(catalog_id, float(score), box, raw_id))\n            return DetectionResult(self._version, predictions,\n                                   self._mapping.sha256 if self._mapping else None,self.catalog.sha256)\n', 'ai_engine/prepare_detection_data.py': '"""Convert existing GTSRB ROI annotations into a reproducible YOLO bootstrap dataset.\n\nThis is a pipeline baseline, not a Vietnamese full-scene detection benchmark.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport shutil\nimport tempfile\nfrom collections import Counter\nfrom pathlib import Path\n\nfrom PIL import Image\nimport yaml\n\nfrom ai_engine.config import CLASS_MAPPING, PROJECT_ROOT\n\nNAMES = [CLASS_MAPPING[i]["qcvn_code"] for i in range(10)]\n\n\ndef build_dataset(archive, csv_root, output, limit_per_class=None, roi_max_inclusive=False, deduplicate_overlaps=False):\n    archive, csv_root, output = Path(archive).resolve(), Path(csv_root).resolve(), Path(output).resolve()\n    if output.exists():\n        raise ValueError("Output already exists; choose a new directory to preserve the previous dataset")\n    if limit_per_class is not None and limit_per_class < 1:\n        raise ValueError("limit_per_class must be positive")\n    rows_by_split, tracks, paths, hashes, overlaps = {}, {}, {}, {}, []\n    for split in ("train", "val", "test"):\n        with (csv_root / f"{split}.csv").open(encoding="utf-8-sig", newline="") as source:\n            rows = list(csv.DictReader(source))\n        counts, selected = Counter(), []\n        for row in rows:\n            class_id = int(row["NewClassId"])\n            if class_id not in CLASS_MAPPING or row["QCVN_Code"] != NAMES[class_id]:\n                raise ValueError("CSV class ordering does not match the project taxonomy")\n            path = (archive / row["Path"]).resolve()\n            if not path.is_relative_to(archive) or not path.is_file():\n                raise ValueError(f"Missing or unsafe source image: {row[\'Path\']}")\n            # Check all rows for train/val track leakage, even in a limited smoke subset.\n            track = row.get("CompoundTrackId")\n            if split in ("train", "val") and not track:\n                raise ValueError("Train/val CSVs need CompoundTrackId to prevent track leakage")\n            if track:\n                if track in tracks and tracks[track] != split:\n                    raise ValueError(f"Track leakage across splits: {track}")\n                tracks[track] = split\n            if path in paths:\n                raise ValueError(f"Repeated image path: {row[\'Path\']}")\n            paths[path] = split\n            if limit_per_class is not None and counts[class_id] >= limit_per_class:\n                continue\n            with Image.open(path) as image:\n                width, height = image.size\n                if (width, height) != (int(row["Width"]), int(row["Height"])):\n                    raise ValueError(f"Image dimensions disagree with CSV: {row[\'Path\']}")\n                image.verify()\n            x1, y1, x2, y2 = [int(row[key]) for key in ("Roi.X1", "Roi.Y1", "Roi.X2", "Roi.Y2")]\n            if roi_max_inclusive:\n                x2, y2 = x2 + 1, y2 + 1\n            if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):\n                raise ValueError(f"Invalid ROI: {row[\'Path\']}")\n            checksum = hashlib.sha256(path.read_bytes()).hexdigest()\n            if checksum in hashes and hashes[checksum] != split:\n                if not deduplicate_overlaps:\n                    raise ValueError(f"Identical image content across splits: {row[\'Path\']}; inspect data or pass --deduplicate-overlaps")\n                overlaps.append({"sha256":checksum,"previous_split":hashes[checksum],"split":split,"source":row["Path"]})\n            hashes[checksum] = split\n            label = f"{class_id} {(x1+x2)/(2*width):.8f} {(y1+y2)/(2*height):.8f} {(x2-x1)/width:.8f} {(y2-y1)/height:.8f}\\n"\n            selected.append((path, row, checksum, label))\n            counts[class_id] += 1\n        if set(counts) != set(range(10)):\n            raise ValueError(f"{split} must contain all 10 classes")\n        rows_by_split[split] = selected\n    dropped = []\n    if deduplicate_overlaps:\n        # Preserve the independent test set. Remove its exact copies from val/train,\n        # and val\'s exact copies from train. Never modify original CSVs or images.\n        for split, selected in rows_by_split.items():\n            retained = []\n            for entry in selected:\n                if hashes[entry[2]] != split:\n                    dropped.append({"source":entry[1]["Path"],"split":split,"sha256":entry[2]})\n                else:\n                    retained.append(entry)\n            if {int(entry[1]["NewClassId"]) for entry in retained} != set(range(10)):\n                raise ValueError(f"Deduplication leaves {split} without examples of every class")\n            rows_by_split[split] = retained\n    output.parent.mkdir(parents=True, exist_ok=True)\n    staging = Path(tempfile.mkdtemp(prefix=".yolo-build-", dir=output.parent))\n    try:\n        manifest = {"kind": "gtsrb_roi_bootstrap", "domain_limit": "Cropped German signs with ROI; not full-scene Vietnamese road data",\n                    "roi_max": "inclusive" if roi_max_inclusive else "exclusive (matches existing CNN crop pipeline)",\n                    "limit_per_class": limit_per_class, "names": NAMES, "splits": {},\n                    "deduplication": {"enabled":deduplicate_overlaps,"priority":"test > val > train", "overlaps":overlaps,"dropped":dropped}}\n        for split, rows in rows_by_split.items():\n            image_dir, label_dir = staging / "images" / split, staging / "labels" / split\n            image_dir.mkdir(parents=True)\n            label_dir.mkdir(parents=True)\n            entries = []\n            for index, (source, row, checksum, label) in enumerate(rows):\n                filename = f"{index:06d}_{source.stem}{source.suffix.lower()}"\n                shutil.copyfile(source, image_dir / filename)\n                (label_dir / Path(filename).with_suffix(".txt")).write_text(label, encoding="utf-8")\n                entries.append({"file": filename, "source": row["Path"], "sha256": checksum,\n                                "class_id": int(row["NewClassId"]), "track": row.get("CompoundTrackId"), "label": label.strip()})\n            manifest["splits"][split] = entries\n        (staging / "data.yaml").write_text(yaml.safe_dump(\n            {"path": str(output), "train": "images/train", "val": "images/val", "test": "images/test", "names": dict(enumerate(NAMES))},\n            allow_unicode=True, sort_keys=False), encoding="utf-8")\n        (staging / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")\n        staging.rename(output)\n    except Exception:\n        shutil.rmtree(staging)\n        raise\n    return {split: len(rows) for split, rows in rows_by_split.items()}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--archive", type=Path, default=Path(PROJECT_ROOT) / "archive")\n    parser.add_argument("--csv-root", type=Path, default=Path(PROJECT_ROOT) / "data/processed")\n    parser.add_argument("--output", type=Path, default=Path(PROJECT_ROOT) / "data/yolo_bootstrap")\n    parser.add_argument("--limit-per-class", type=int)\n    parser.add_argument("--roi-max-inclusive", action="store_true", help="Treat CSV ROI.X2/Y2 as inclusive; default matches existing crop code")\n    parser.add_argument("--deduplicate-overlaps", action="store_true", help="Remove exact cross-split copies from train/val, preserve test, record exclusions")\n    args = parser.parse_args()\n    try:\n        print(json.dumps(build_dataset(args.archive, args.csv_root, args.output, args.limit_per_class, args.roi_max_inclusive, args.deduplicate_overlaps), indent=2))\n    except (ValueError, OSError, KeyError) as error:\n        parser.exit(1, f"Dataset preparation failed: {error}\\n")\n\n\nif __name__ == "__main__":\n    main()\n', 'ai_engine/train_detection.py': '"""Train/evaluate YOLO with dataset validation, fixed taxonomy and recorded provenance."""\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport yaml\nfrom PIL import Image\n\nfrom ai_engine.config import PROJECT_ROOT\nfrom ai_engine.detection import checkpoint_hash, configure_ultralytics\nfrom ai_engine.prepare_detection_data import NAMES\n\n\ndef validate_dataset(data):\n    data = Path(data).resolve()\n    config = yaml.safe_load(data.read_text(encoding="utf-8"))\n    names = config.get("names")\n    if isinstance(names, dict):\n        if set(names) != set(range(10)):\n            raise ValueError("data.yaml must define exactly 10 indexed names")\n        names = [names.get(i) for i in range(10)]\n    if names != NAMES:\n        raise ValueError("data.yaml must use the exact 10 project names in order")\n    root = Path(config.get("path", data.parent))\n    root = (data.parent / root).resolve() if not root.is_absolute() else root.resolve()\n    checksum = hashlib.sha256(data.read_bytes())\n    splits, seen_paths, seen_hashes = {}, set(), {}\n    for split in ("train", "val", "test"):\n        value = config.get(split)\n        if not isinstance(value, str):\n            raise ValueError("Each split must point to an images directory (including independent test)")\n        directory = (root / value).resolve()\n        if not directory.is_relative_to(root) or directory.parent.name != "images":\n            raise ValueError("Split paths must stay under dataset root, using images/<split> layout")\n        images = sorted(path for path in directory.rglob("*") if path.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp"))\n        if not images:\n            raise ValueError(f"Empty image split: {split}")\n        instances, classes = 0, set()\n        for image in images:\n            resolved = image.resolve()\n            if not resolved.is_relative_to(root) or resolved in seen_paths:\n                raise ValueError("Unsafe or repeated image path across splits")\n            seen_paths.add(resolved)\n            content_hash = hashlib.sha256(image.read_bytes()).hexdigest()\n            if content_hash in seen_hashes and seen_hashes[content_hash] != split:\n                raise ValueError("Identical image content across train/val/test")\n            seen_hashes[content_hash] = split\n            label = root / "labels" / directory.name / image.relative_to(directory).with_suffix(".txt")\n            if not label.resolve().is_relative_to(root) or not label.is_file():\n                raise ValueError(f"Missing label (use empty file for a verified negative): {label}")\n            text = label.read_text(encoding="utf-8")\n            for line in text.splitlines():\n                if not line.strip():\n                    continue\n                parts = line.split()\n                if len(parts) != 5:\n                    raise ValueError(f"Expected class x_center y_center width height: {label}")\n                class_id = int(parts[0])\n                x, y, w, h = map(float, parts[1:])\n                if class_id not in range(10) or not all(math.isfinite(n) for n in (x,y,w,h)) or not (0 < w <= 1 and 0 < h <= 1):\n                    raise ValueError(f"Invalid YOLO label: {label}")\n                if x-w/2 < -1e-6 or y-h/2 < -1e-6 or x+w/2 > 1+1e-6 or y+h/2 > 1+1e-6:\n                    raise ValueError(f"Box exceeds image bounds: {label}")\n                instances += 1\n                classes.add(class_id)\n            with Image.open(image) as source:\n                if source.width * source.height > 40_000_000:\n                    raise ValueError(f"Image too large: {image}")\n                source.verify()\n            checksum.update(f"{split}:{image.relative_to(root)}:{content_hash}:{text}".encode())\n        if classes != set(range(10)):\n            raise ValueError(f"{split} must include annotated examples of all 10 classes")\n        splits[split] = {"images": len(images), "instances": instances}\n        print(f"Validated {split}: {len(images)} images, {instances} boxes", flush=True)\n    manifest = root / "manifest.json"\n    domain = json.loads(manifest.read_text(encoding="utf-8")).get("kind", "custom") if manifest.is_file() else "custom_full_scene_unverified"\n    return {"yaml": str(data), "sha256": checksum.hexdigest(), "domain": domain, "splits": splits}\n\n\ndef metrics_report(metrics):\n    # ap_class_index includes only classes with ground-truth instances; never invent per-class scores.\n    box = metrics.box\n    values = {int(class_id): {"precision": float(box.p[index]), "recall": float(box.r[index]),\n                            "map50": float(box.ap50[index]), "map50_95": float(box.ap[index])}\n              for index, class_id in enumerate(box.ap_class_index)}\n    return {"precision": float(box.mp), "recall": float(box.mr), "map50": float(box.map50),\n            "map50_95": float(box.map), "per_class": {name: values.get(i) for i, name in enumerate(NAMES)}}\n\n\ndef activate(weights, destination):\n    destination = Path(destination).resolve()\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    # Never destroy the previous checkpoint. Stage the new file before replacing.\n    if destination.exists():\n        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")\n        shutil.copy2(destination, destination.with_name(f"best.{stamp}.pt"))\n    staging = destination.with_suffix(".pt.pending")\n    shutil.copy2(weights, staging)\n    os.replace(staging, destination)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("command", choices=("train", "evaluate", "validate"))\n    parser.add_argument("--data", type=Path, required=True)\n    parser.add_argument("--model", help="Local checkpoint, or Ultralytics model architecture such as yolo11n.yaml")\n    parser.add_argument("--allow-download", action="store_true", help="Allow Ultralytics to download official starting weights")\n    parser.add_argument("--epochs", type=int, default=50)\n    parser.add_argument("--imgsz", type=int, default=640)\n    parser.add_argument("--batch", type=int, default=8)\n    parser.add_argument("--device", default="0")\n    parser.add_argument("--workers", type=int, default=0, help="0 is the safe Windows default")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--output", type=Path)\n    parser.add_argument("--smoke", action="store_true", help="Mark a pipeline check; cannot activate its checkpoint")\n    parser.add_argument("--activate", action="store_true", help="Copy a trained detector to the API checkpoint location")\n    args = parser.parse_args()\n    if args.command == "validate":\n        try:\n            print(json.dumps(validate_dataset(args.data), indent=2, ensure_ascii=False))\n            return\n        except (ValueError, OSError, KeyError, TypeError) as error:\n            parser.exit(1, f"Dataset validation failed: {error}\\n")\n    if args.model is None or args.output is None:\n        parser.error("Train/evaluate require --model and --output")\n    if args.activate and (args.smoke or args.command != "train"):\n        parser.error("Only a non-smoke training run can activate weights")\n    if args.epochs < 1 or args.imgsz < 32 or args.batch < 1 or args.workers < 0:\n        parser.error("Invalid training dimensions or counts")\n    if args.output.exists():\n        parser.error("Output exists; select a new run directory")\n    if args.model.lower().endswith(".pt") and not Path(args.model).is_file() and not args.allow_download:\n        parser.error("Checkpoint missing; supply a local file or explicitly pass --allow-download")\n    if args.command == "evaluate" and not Path(args.model).is_file():\n        parser.error("Evaluation requires a local trained checkpoint")\n    try:\n        dataset = validate_dataset(args.data)\n        configure_ultralytics()\n        from ultralytics import YOLO, __version__\n        import torch\n        model = YOLO(args.model, task="detect")\n        output = args.output.resolve()\n        if args.command == "train":\n            model.train(data=str(args.data.resolve()), epochs=args.epochs, imgsz=args.imgsz, batch=args.batch,\n                        device=args.device, workers=args.workers, seed=args.seed, deterministic=True,\n                        fliplr=0.0, flipud=0.0, degrees=5.0, project=str(output.parent), name=output.name,\n                        exist_ok=False, plots=True, amp=False, pretrained=not args.model.endswith(".yaml"))\n            weights = Path(model.trainer.best)\n            model = YOLO(str(weights), task="detect")\n        else:\n            output.mkdir(parents=True)\n            weights = Path(args.model).resolve()\n        ordered_names = [model.names[i] for i in range(len(model.names))]\n        if ordered_names != NAMES or model.task != "detect":\n            raise ValueError("Trained checkpoint taxonomy does not match SignSight")\n        evaluations = {}\n        for split in ("val", "test"):\n            metrics = model.val(data=str(args.data.resolve()), split=split, imgsz=args.imgsz, batch=args.batch,\n                                device=args.device, workers=args.workers, plots=True,\n                                project=str(output), name=f"evaluate_{split}", exist_ok=False)\n            evaluations[split] = metrics_report(metrics)\n        report = {"created_at": datetime.now(timezone.utc).isoformat(), "smoke": args.smoke,\n                  "dataset": dataset, "weights": str(weights), "model_version": checkpoint_hash(weights),\n                  "ultralytics": __version__, "torch": torch.__version__, "settings": {key: str(value) if isinstance(value, Path) else value for key,value in vars(args).items()},\n                  "metrics": evaluations,\n                  "limitation": "GTSRB ROI results measure bootstrap detection; Vietnamese full-scene quality needs an independent road-scene dataset" if dataset["domain"] == "gtsrb_roi_bootstrap" else "Verify independent scene/session splits and deployment domain before accepting metrics"}\n        (output / "detection_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n        if args.activate:\n            activate(weights, Path(PROJECT_ROOT) / "ai_engine/checkpoints/detection/best.pt")\n        print(f"Report: {output / \'detection_report.json\'}")\n    except (ValueError, OSError, ImportError) as error:\n        parser.exit(1, f"Detection pipeline failed: {error}\\n")\n\n\nif __name__ == "__main__":\n    main()\n', 'ai_engine/notebook_data.py': '"""Reusable notebook dataset cache; Kaggle/Drive access stays in the notebook."""\nimport base64\nimport gzip\nimport hashlib\nimport json\nimport stat\nimport tempfile\nimport zipfile\nfrom pathlib import Path\n\nimport yaml\n\nfrom ai_engine.prepare_detection_data import NAMES, build_dataset\nfrom ai_engine.train_detection import validate_dataset\n\nSPLITS = (\'train.csv\', \'val.csv\', \'test.csv\')\n\n\nclass DatasetCacheMiss(ValueError):\n    """No compatible cache exists and a raw source is needed."""\n\n\ndef _sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as source:\n        for chunk in iter(lambda: source.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef resolve_archive(local_roots, source_handle, download):\n    def find(roots):\n        candidates = set()\n        for root in map(Path, roots):\n            if root.is_dir():\n                for annotation in root.rglob(\'Train.csv\'):\n                    if (annotation.parent / \'Test.csv\').is_file():\n                        candidates.add(annotation.parent.resolve())\n        if len(candidates) > 1:\n            raise ValueError(\'Multiple GTSRB sources; set RAW_DATASET_DIR to one archive\')\n        return next(iter(candidates), None)\n    existing = find(local_roots)\n    if existing is not None:\n        return existing\n    downloaded = find([download(source_handle)])\n    if downloaded is None:\n        raise ValueError(\'Downloaded raw source lacks Train.csv and Test.csv\')\n    return downloaded\n\n\ndef restore_splits(payload, checksums, target):\n    if set(payload) != set(SPLITS) or set(checksums) != set(SPLITS):\n        raise ValueError(\'Expected exactly train.csv, val.csv and test.csv\')\n    decoded = {}\n    for name in SPLITS:\n        data = gzip.decompress(base64.b64decode(payload[name], validate=True))\n        if hashlib.sha256(data).hexdigest() != checksums[name]:\n            raise ValueError(f\'Embedded split checksum mismatch: {name}\')\n        decoded[name] = data\n    target = Path(target)\n    target.mkdir(parents=True, exist_ok=True)\n    for name, data in decoded.items():\n        (target / name).write_bytes(data)\n    return target\n\n\ndef _descriptor(csv_root, source_id):\n    directory = Path(__file__).parent\n    return {\'schema\': 1, \'source\': source_id, \'names\': NAMES,\n            \'splits\': {name: _sha(Path(csv_root) / name) for name in SPLITS},\n            \'code\': {name: _sha(directory / name) for name in\n                     (\'notebook_data.py\', \'prepare_detection_data.py\', \'train_detection.py\')},\n            \'deduplicate_overlaps\': True, \'roi_max_inclusive\': False}\n\n\ndef _validate_cache(root, descriptor):\n    metadata = json.loads((root / \'_signsight_cache.json\').read_text(encoding=\'utf-8\'))\n    if metadata[\'descriptor\'] != descriptor:\n        raise ValueError(\'Dataset cache descriptor mismatch\')\n    config = yaml.safe_load((root / \'data.yaml\').read_text(encoding=\'utf-8\'))\n    if config.get(\'path\') != \'.\':\n        raise ValueError(\'Dataset cache must have portable path: .\')\n    if _sha(root / \'manifest.json\') != metadata[\'manifest_sha256\']:\n        raise ValueError(\'Dataset cache manifest checksum mismatch\')\n    report = validate_dataset(root / \'data.yaml\')\n    if report[\'sha256\'] != metadata[\'sha256\']:\n        raise ValueError(\'Dataset cache checksum mismatch\')\n    return report\n\n\ndef _safe_extract(bundle, target):\n    entries = bundle.infolist()\n    if len(entries) > 100000 or sum(entry.file_size for entry in entries) > 8 * 1024**3:\n        raise ValueError(\'Dataset cache ZIP exceeds size limits\')\n    seen = set()\n    for entry in entries:\n        path = (target / entry.filename).resolve()\n        if (not path.is_relative_to(target.resolve()) or path == target.resolve()\n                or \'\\\\\' in entry.filename or \':\' in entry.filename\n                or stat.S_ISLNK(entry.external_attr >> 16) or path in seen):\n            raise ValueError(\'Unsafe dataset cache ZIP path\')\n        seen.add(path)\n    bundle.extractall(target)\n\n\ndef _persist(root, saved, key, descriptor, report):\n    saved.mkdir(parents=True, exist_ok=True)\n    archive, record = saved / f\'{key}.zip\', saved / f\'{key}.json\'\n    if archive.exists() or record.exists():\n        if not archive.is_file() or not record.is_file():\n            raise ValueError(\'Incomplete persistent cache; choose a new cache folder\')\n        metadata = json.loads(record.read_text(encoding=\'utf-8\'))\n        if (metadata[\'descriptor\'] != descriptor or metadata[\'sha256\'] != report[\'sha256\']\n                or metadata[\'archive_sha256\'] != _sha(archive)):\n            raise ValueError(\'Persistent cache checksum mismatch\')\n        return\n    # Only this task\'s temporary directory is cleaned up. Existing cache is preserved.\n    with tempfile.TemporaryDirectory(prefix=\'.signsight-save-\', dir=saved) as temporary:\n        staging = Path(temporary)\n        packed = staging / \'dataset.zip\'\n        with zipfile.ZipFile(packed, \'w\', compression=zipfile.ZIP_DEFLATED) as bundle:\n            for path in sorted(root.rglob(\'*\')):\n                if path.is_file():\n                    bundle.write(path, path.relative_to(root).as_posix())\n        metadata = {\'descriptor\': descriptor, \'sha256\': report[\'sha256\'], \'archive_sha256\': _sha(packed)}\n        pending = staging / \'metadata.json\'\n        pending.write_text(json.dumps(metadata, sort_keys=True), encoding=\'utf-8\')\n        packed.rename(archive)\n        pending.rename(record)\n\n\ndef prepare_cached_dataset(archive, csv_root, local_cache_root, *, source_id, persistent_root=None):\n    descriptor = _descriptor(csv_root, source_id)\n    key = hashlib.sha256(json.dumps(descriptor, sort_keys=True).encode()).hexdigest()\n    prepared = Path(local_cache_root) / \'prepared\'\n    prepared.mkdir(parents=True, exist_ok=True)\n    root = prepared / key\n    saved = Path(persistent_root) if persistent_root is not None else None\n    if root.exists():\n        report = _validate_cache(root, descriptor)\n    else:\n        with tempfile.TemporaryDirectory(prefix=\'.signsight-prepare-\', dir=prepared) as temporary:\n            staging = Path(temporary) / \'dataset\'\n            packed = saved / f\'{key}.zip\' if saved is not None else None\n            record = saved / f\'{key}.json\' if saved is not None else None\n            if packed is not None and (packed.exists() or record.exists()):\n                if not packed.is_file() or not record.is_file():\n                    raise ValueError(\'Incomplete persistent cache; choose a new cache folder\')\n                metadata = json.loads(record.read_text(encoding=\'utf-8\'))\n                if metadata[\'descriptor\'] != descriptor or metadata[\'archive_sha256\'] != _sha(packed):\n                    raise ValueError(\'Persistent cache checksum mismatch\')\n                staging.mkdir()\n                with zipfile.ZipFile(packed) as bundle:\n                    _safe_extract(bundle, staging)\n                report = _validate_cache(staging, descriptor)\n                if report[\'sha256\'] != metadata[\'sha256\']:\n                    raise ValueError(\'Persistent cache dataset checksum mismatch\')\n            else:\n                if archive is None:\n                    raise DatasetCacheMiss(\'No compatible cache; raw source required\')\n                build_dataset(archive, csv_root, staging, deduplicate_overlaps=True)\n                data = staging / \'data.yaml\'\n                config = yaml.safe_load(data.read_text(encoding=\'utf-8\'))\n                config[\'path\'] = \'.\'\n                data.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding=\'utf-8\')\n                report = validate_dataset(data)\n                (staging / \'_signsight_cache.json\').write_text(json.dumps({\n                    \'descriptor\': descriptor, \'sha256\': report[\'sha256\'],\n                    \'manifest_sha256\': _sha(staging / \'manifest.json\')}, sort_keys=True), encoding=\'utf-8\')\n            staging.rename(root)\n        report[\'yaml\'] = str(root / \'data.yaml\')\n    if saved is not None:\n        _persist(root, saved, key, descriptor, report)\n    return root / \'data.yaml\', report\n', 'data/taxonomy.json': '[\n  {\n    "new_id": 0,\n    "gtsrb_id": 2,\n    "qcvn_code": "P.127 (50)",\n    "name_vi": "Tốc độ tối đa cho phép (50 km/h)",\n    "name_en": "Speed limit (50 km/h)",\n    "category": "Biển báo cấm (Prohibitory)",\n    "vietnam_context": "Rất phổ biến trong khu vực đô thị, khu đông dân cư đường 2 chiều không có dải phân cách giữa.",\n    "shape": "circle",\n    "border_color": "red",\n    "background_color": "white"\n  },\n  {\n    "new_id": 1,\n    "gtsrb_id": 3,\n    "qcvn_code": "P.127 (60)",\n    "name_vi": "Tốc độ tối đa cho phép (60 km/h)",\n    "name_en": "Speed limit (60 km/h)",\n    "category": "Biển báo cấm (Prohibitory)",\n    "vietnam_context": "Phổ biến trong đô thị có dải phân cách giữa và các trục đường đôi tại Việt Nam.",\n    "shape": "circle",\n    "border_color": "red",\n    "background_color": "white"\n  },\n  {\n    "new_id": 2,\n    "gtsrb_id": 5,\n    "qcvn_code": "P.127 (80)",\n    "name_vi": "Tốc độ tối đa cho phép (80 km/h)",\n    "name_en": "Speed limit (80 km/h)",\n    "category": "Biển báo cấm (Prohibitory)",\n    "vietnam_context": "Áp dụng ngoài khu đông dân cư, đường vành đai, quốc lộ và một số đoạn đường cao tốc.",\n    "shape": "circle",\n    "border_color": "red",\n    "background_color": "white"\n  },\n  {\n    "new_id": 3,\n    "gtsrb_id": 9,\n    "qcvn_code": "P.125",\n    "name_vi": "Cấm vượt",\n    "name_en": "No passing / No overtaking",\n    "category": "Biển báo cấm (Prohibitory)",\n    "vietnam_context": "Biển cấm các loại xe cơ giới vượt nhau trên các đoạn đường hẹp, đường đèo dốc nguy hiểm tại Việt Nam.",\n    "shape": "circle",\n    "border_color": "red",\n    "background_color": "white"\n  },\n  {\n    "new_id": 4,\n    "gtsrb_id": 14,\n    "qcvn_code": "P.122",\n    "name_vi": "Dừng lại (STOP)",\n    "name_en": "Stop",\n    "category": "Biển báo cấm (Prohibitory / Safety)",\n    "vietnam_context": "Bắt buộc mọi phương tiện dừng lại trước vạch dừng tại các nút giao nguy hiểm hoặc giao cắt đường sắt.",\n    "shape": "octagon",\n    "border_color": "white",\n    "background_color": "red"\n  },\n  {\n    "new_id": 5,\n    "gtsrb_id": 17,\n    "qcvn_code": "P.102",\n    "name_vi": "Cấm đi ngược chiều",\n    "name_en": "No entry",\n    "category": "Biển báo cấm (Prohibitory)",\n    "vietnam_context": "Cấm tất cả các loại xe đi vào theo chiều đặt biển, đặc biệt tại các tuyến đường một chiều ở Hà Nội, TP.HCM.",\n    "shape": "circle",\n    "border_color": "red",\n    "background_color": "red"\n  },\n  {\n    "new_id": 6,\n    "gtsrb_id": 18,\n    "qcvn_code": "W.245a",\n    "name_vi": "Đi chậm / Nguy hiểm khác",\n    "name_en": "General caution / Danger",\n    "category": "Biển báo nguy hiểm (Warning)",\n    "vietnam_context": "Cảnh báo đoạn đường hay xảy ra tai nạn hoặc có nguy hiểm bất ngờ không có biển báo riêng.",\n    "shape": "triangle",\n    "border_color": "red",\n    "background_color": "yellow_or_white"\n  },\n  {\n    "new_id": 7,\n    "gtsrb_id": 25,\n    "qcvn_code": "W.227",\n    "name_vi": "Công trường đang thi công",\n    "name_en": "Road work",\n    "category": "Biển báo nguy hiểm (Temporary Warning)",\n    "vietnam_context": "Báo trước gần tới đoạn đường đang sửa chữa, thi công. Biển này mang tính tạm thời, rất phù hợp với luồng Revalidation/Expiration.",\n    "shape": "triangle",\n    "border_color": "red",\n    "background_color": "yellow_or_white"\n  },\n  {\n    "new_id": 8,\n    "gtsrb_id": 35,\n    "qcvn_code": "R.301a",\n    "name_vi": "Các xe chỉ được đi thẳng",\n    "name_en": "Ahead only",\n    "category": "Biển hiệu lệnh (Mandatory)",\n    "vietnam_context": "Đặt trước nút giao, bắt buộc các phương tiện chỉ được đi thẳng, không được rẽ trái hay rẽ phải.",\n    "shape": "circle",\n    "border_color": "blue",\n    "background_color": "blue"\n  },\n  {\n    "new_id": 9,\n    "gtsrb_id": 38,\n    "qcvn_code": "R.301e",\n    "name_vi": "Hướng đi phải theo: Đi vòng sang bên phải",\n    "name_en": "Keep right",\n    "category": "Biển hiệu lệnh (Mandatory)",\n    "vietnam_context": "Đặt tại các đảo giao thông, dải phân cách giữa, chướng ngại vật trên đường ở Việt Nam để chỉ dẫn luồng xe chạy.",\n    "shape": "circle",\n    "border_color": "blue",\n    "background_color": "blue"\n  }\n]\n'}
SPLIT_PAYLOAD = {'train.csv': '...', 'val.csv': '...', 'test.csv': '...'}
SPLIT_CHECKSUMS = {'train.csv': '0724492a72957b51096f43f326ee610a1440e996881c0f403d8081bf554447c0', 'val.csv': '60630efccaf1c7833dc93655a1875fa4b535e0b30079177aacdd07ae1cfa32c8', 'test.csv': '8ad2283f450d08b5a3baabd98f60ae33d1be69af74564312170b2be6dfb2f4cf'}
for relative, source in TRAINING_FILES.items():
    target = WORK / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
(WORK / 'ai_engine/__init__.py').touch()
sys.path.insert(0, str(WORK))
os.chdir(WORK)
from ai_engine.notebook_data import DatasetCacheMiss, prepare_cached_dataset, resolve_archive, restore_splits
from ai_engine.train_detection import validate_dataset
CSV_ROOT = restore_splits(SPLIT_PAYLOAD, SPLIT_CHECKSUMS, WORK / 'data/processed')
print('Pipeline ready:', WORK)

## Nguồn dữ liệu và cache
`USE_DRIVE = False`: tải tự động, cache trong runtime; runtime mới có thể phải tải lại.
`USE_DRIVE = True` (khuyên dùng trên Colab): mount Drive, lần đầu lưu một ZIP đã chuẩn bị + checksum;
những phiên sau khôi phục ZIP vào ổ local để train nhanh, không tải raw dataset hay chạy converter lại.
Kaggle: Add Input → GTSRB nếu muốn dùng input có sẵn; Save Version để giữ outputs, `/kaggle/working` không tự giữ qua phiên mới.
Public dataset thường tải được không cần token; nếu Kaggle yêu cầu consent/login, làm theo thông báo KaggleHub.
`RAW_DATASET_DIR` chỉ cần đặt khi có archive riêng. Giữ nguyên `SOURCE_ID` cho nguồn mặc định;
khi đổi nguồn/custom archive, đặt ID phiên bản riêng để tránh dùng nhầm cache.

In [ ]:
USE_DRIVE = False  # đổi True để giữ dataset/checkpoint trên Google Drive qua các phiên Colab
SOURCE_ID = 'meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/versions/1'
RAW_DATASET_DIR = None  # ví dụ '/kaggle/input/gtsrb-german-traffic-sign'
PERSISTENT_CACHE = None
RUNS_ROOT = WORK / 'runs/detection'
if USE_DRIVE:
    assert not IS_KAGGLE, 'Google Drive mount dành cho Colab.'
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/SignSight')
    PERSISTENT_CACHE = DRIVE_ROOT / 'yolo_datasets'
    RUNS_ROOT = DRIVE_ROOT / 'runs/detection'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
CACHE_ROOT = WORK / '.cache/yolo'
try:
    PORTABLE_DATA, DATASET_REPORT = prepare_cached_dataset(
        None, CSV_ROOT, CACHE_ROOT, source_id=SOURCE_ID, persistent_root=PERSISTENT_CACHE)
    print('Reused prepared dataset cache')
except DatasetCacheMiss:
    import kagglehub
    roots = [Path(RAW_DATASET_DIR)] if RAW_DATASET_DIR else [Path('/kaggle/input'), WORK / 'archive']
    if RAW_DATASET_DIR:
        assert Path(RAW_DATASET_DIR).is_dir(), 'RAW_DATASET_DIR không tồn tại.'
        def download_raw(_): raise ValueError('RAW_DATASET_DIR thiếu Train.csv/Test.csv')
    else:
        download_raw = kagglehub.dataset_download
    archive_root = resolve_archive(roots, SOURCE_ID, download_raw)
    PORTABLE_DATA, DATASET_REPORT = prepare_cached_dataset(
        archive_root, CSV_ROOT, CACHE_ROOT, source_id=SOURCE_ID, persistent_root=PERSISTENT_CACHE)
DATASET_ROOT = PORTABLE_DATA.parent
# Ultralytics resolves relative path via its datasets_dir; supply an absolute runtime path.
# Keep cached data.yaml portable so a new Colab runtime has the same fingerprint.
import yaml
config = yaml.safe_load(PORTABLE_DATA.read_text(encoding='utf-8'))
config['path'] = str(DATASET_ROOT.resolve())
DATA = WORK / 'data_runtime.yaml'
DATA.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
REPORT_PATH = WORK / 'dataset_report.json'
REPORT_PATH.write_text(json.dumps(DATASET_REPORT, indent=2, ensure_ascii=False), encoding='utf-8')
print(' | '.join(
    f"{split}: {counts['images']:,} images / {counts['instances']:,} boxes"
    for split, counts in DATASET_REPORT['splits'].items()))
print('Full validation report: dataset_report.json')

## Kiểm tra nhãn trước khi train
Hiển thị annotation mẫu; kiểm tra class và ROI trước khi dùng GPU.
Không dùng test để chọn model/threshold. Deduplication giữ test > val > train và ghi các bản sao bị loại vào manifest.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from ai_engine.prepare_detection_data import NAMES
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for axis, source in zip(axes.flat, sorted((DATASET_ROOT/'images/train').glob('*'))[:6]):
    image = Image.open(source).convert('RGB')
    draw = ImageDraw.Draw(image)
    label = DATASET_ROOT/'labels/train'/source.with_suffix('.txt').name
    for line in label.read_text().splitlines():
        if not line.strip(): continue
        cls, x, y, w, h = map(float, line.split())
        box = ((x-w/2)*image.width, (y-h/2)*image.height, (x+w/2)*image.width, (y+h/2)*image.height)
        draw.rectangle(box, outline='yellow', width=2)
        draw.text((box[0], box[1]), NAMES[int(cls)], fill='yellow')
    axis.imshow(image); axis.axis('off')
plt.tight_layout()

## Train + đánh giá val/test
Mặc định fine-tune YOLO11n COCO trên taxonomy 10 lớp, 50 epochs. Nếu GPU thiếu bộ nhớ giảm BATCH.
RUN_DIR nên nằm trên Drive khi train lâu; checkpoint tự lưu ở mỗi epoch.
Pipeline tắt flip ngang/dọc vì biển chỉ hướng; ghi seed, hashes, phiên bản, precision/recall/mAP theo lớp.
Run đã có bị từ chối để giữ kết quả trước. Test chỉ đánh giá sau khi chọn best theo val.

In [ ]:
EPOCHS = 50
IMAGE_SIZE = 640
BATCH = 8
from datetime import datetime, timezone
RUN_DIR = RUNS_ROOT / datetime.now(timezone.utc).strftime('yolo_%Y%m%dT%H%M%S%fZ')  # có thể đổi sang Drive
SMOKE = False  # True cho thử pipeline 1 epoch; không dùng checkpoint smoke để triển khai
args = [sys.executable, '-m', 'ai_engine.train_detection', 'train', '--data', str(DATA),
        '--model', 'yolo11n.pt', '--allow-download', '--epochs', str(1 if SMOKE else EPOCHS),
        '--imgsz', str(IMAGE_SIZE), '--batch', str(BATCH), '--device', '0', '--workers', '2',
        '--output', str(RUN_DIR)]
if SMOKE: args.append('--smoke')
subprocess.check_call(args)
report = json.loads((RUN_DIR/'detection_report.json').read_text())
print(json.dumps(report['metrics'], indent=2, ensure_ascii=False))

## Xem lỗi detection
Các bảng metric và confusion matrix nằm trong thư mục run. Thử cả ảnh nền, 0/1/nhiều biển và biển ngoài scope; đây là kiểm tra riêng ngoài benchmark ROI.

In [ ]:
from IPython.display import display
for split in ('val', 'test'):
    plot = RUN_DIR / f'evaluate_{split}/confusion_matrix.png'
    if plot.is_file(): display(Image.open(plot))
from ultralytics import YOLO
best = YOLO(str(RUN_DIR/'weights/best.pt'))
samples = sorted((DATASET_ROOT/'images/test').glob('*'))[:4]
for result in best.predict([str(path) for path in samples], conf=0.25, save=False):
    display(Image.fromarray(result.plot()[..., ::-1]))

## Tải kết quả về local
ZIP gồm best.pt, last.pt, detection_report.json, results.csv, config train, plots và data.yaml/manifest.
Giải nén, đặt best.pt vào ai_engine/checkpoints/detection/best.pt; cài requirements-yolo.txt trên server rồi restart.
Giữ detection_report.json cùng checkpoint. Chưa kết luận chất lượng cảnh đường phố nếu train bằng GTSRB ROI.

In [ ]:
EXPORT = WORK / (RUN_DIR.name + '_result')
assert not EXPORT.exists(), 'Export tồn tại; chọn tên khác để giữ bản trước.'
shutil.copytree(RUN_DIR, EXPORT/'run')
(EXPORT/'dataset').mkdir()
shutil.copy2(PORTABLE_DATA, EXPORT/'dataset/data.yaml')
manifest = DATASET_ROOT/'manifest.json'
if manifest.is_file(): shutil.copy2(manifest, EXPORT/'dataset/manifest.json')
archive = shutil.make_archive(str(EXPORT), 'zip', EXPORT)
if IS_KAGGLE:
    from IPython.display import FileLink
    display(FileLink(archive))
else:
    from google.colab import files
    files.download(archive)